# PDF Metadata for ETU TORUS Source Document Exploration with `mlcroissant`
This notebook provides a template-based walkthrough for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qrqx-ry0g/fair2.json`

### Notes
- This notebook follows the requested template structure and style.
- All dataset entities are referenced by their Croissant `@id` values whenever available.
- The provided FAIR package summary indicates that this dataset is document-centric and may contain metadata, provenance, and documentary content derived from ETU TORUS source PDFs.
- Some Croissant datasets expose record sets directly, while others are metadata-heavy and may have limited tabular record access. The notebook therefore includes robust inspection code that adapts to what is available at runtime.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`, then inspect high-level metadata fields without treating `dataset.metadata` like a dictionary object.

In [ ]:
import json
import math
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qrqx-ry0g/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON for inspection
metadata_json = dataset.metadata.to_json()

dataset_name = metadata_json.get("name", "Unknown dataset")
dataset_description = metadata_json.get("description", "No description available.")
dataset_id = metadata_json.get("@id", "Unknown @id")
dataset_version = metadata_json.get("version", "Unknown version")

print(f"Dataset name: {dataset_name}")
print(f"Dataset @id: {dataset_id}")
print(f"Version: {dataset_version}")
print(f"Description: {dataset_description}")

In [ ]:
# Display a compact metadata summary
summary_fields = [
    "identifier",
    "license",
    "datePublished",
    "keywords",
    "citeAs",
    "temporalCoverage",
    "distribution"
]

for field_name in summary_fields:
    print(f"\n{field_name}:")
    pprint(metadata_json.get(field_name, None))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Because Croissant datasets can vary in structure, we first inspect the metadata JSON and then build helper variables dynamically from discovered `@id` values.

In [ ]:
# Inspect top-level metadata keys
print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

In [ ]:
# Extract record sets from metadata JSON.
# We reference all discovered entities by their @id values.
record_sets_meta = metadata_json.get("recordSet", []) or []

print(f"Number of record sets declared in metadata: {len(record_sets_meta)}")

record_set_ids = []
record_set_summaries = []

for rs in record_sets_meta:
    rs_id = rs.get("@id", "<missing-record-set-id>")
    rs_name = rs.get("name", "<unnamed>")
    rs_desc = rs.get("description", "")
    fields = rs.get("field", []) or []
    field_ids = [f.get("@id", "<missing-field-id>") for f in fields]
    record_set_ids.append(rs_id)
    record_set_summaries.append({
        "record_set_@id": rs_id,
        "record_set_name": rs_name,
        "n_fields": len(fields),
        "field_@ids": field_ids,
        "description": rs_desc,
    })

record_sets_df = pd.DataFrame(record_set_summaries)
record_sets_df

The provided FAIR package excerpt suggests that `recordSet` may be empty for this resource. The next cells still preserve the same workflow, but add safe fallbacks so the notebook remains useful even when no tabular record set is declared.

In [ ]:
# Show detailed field-level information for each record set, if any exist
field_rows = []

for rs in record_sets_meta:
    rs_id = rs.get("@id", "<missing-record-set-id>")
    for field in rs.get("field", []) or []:
        field_rows.append({
            "record_set_@id": rs_id,
            "field_@id": field.get("@id", "<missing-field-id>"),
            "field_name": field.get("name", "<unnamed-field>"),
            "description": field.get("description", ""),
            "dataType": field.get("dataType", None),
            "source": field.get("source", None),
        })

fields_df = pd.DataFrame(field_rows)
fields_df

In [ ]:
# Example template-style record iteration using discovered @id values.
# This cell only runs when at least one record set is available.
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Previewing records from record set @id: {example_record_set_id}")
    
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(x)
        if i >= 2:
            break
else:
    print("No record sets were declared in metadata, so no record iteration can be demonstrated for this dataset.")

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame using record set `@id` values discovered from the metadata.

If no record sets are available, we create a compact metadata DataFrame from selected top-level dataset attributes so that the notebook still supports downstream exploration.

In [ ]:
# Extract data from each record set using @id references
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set {record_set_id} with shape {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

if not dataframes:
    print("No record-set DataFrames were created. Building a fallback metadata DataFrame.")
    fallback_rows = [
        {"attribute_@id": "@id", "value": metadata_json.get("@id")},
        {"attribute_@id": "name", "value": metadata_json.get("name")},
        {"attribute_@id": "description", "value": metadata_json.get("description")},
        {"attribute_@id": "identifier", "value": metadata_json.get("identifier")},
        {"attribute_@id": "version", "value": metadata_json.get("version")},
        {"attribute_@id": "datePublished", "value": metadata_json.get("datePublished")},
        {"attribute_@id": "license", "value": metadata_json.get("license")},
        {"attribute_@id": "citeAs", "value": metadata_json.get("citeAs")},
        {"attribute_@id": "temporalCoverage", "value": metadata_json.get("temporalCoverage")},
        {"attribute_@id": "keywords", "value": ", ".join(metadata_json.get("keywords", []))},
    ]
    fallback_record_set_id = "fallback:dataset_metadata"
    dataframes[fallback_record_set_id] = pd.DataFrame(fallback_rows)
else:
    fallback_record_set_id = None

available_df_ids = list(dataframes.keys())
print("\nAvailable DataFrame IDs:")
print(available_df_ids)

In [ ]:
# Inspect the first available DataFrame
selected_record_set_id = available_df_ids[0]
print(f"Selected DataFrame @id: {selected_record_set_id}")
print("Columns:")
print(dataframes[selected_record_set_id].columns.tolist())

dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply practical data processing steps using whichever DataFrame is available.

The template calls for filtering, normalization, and grouping. Since this dataset may or may not expose numeric fields in record sets, the code below dynamically identifies numeric columns where possible. If no numeric columns exist, it performs metadata-oriented exploration instead.

In [ ]:
eda_df = dataframes[selected_record_set_id].copy()

# Attempt numeric conversion for object columns where possible
for col in eda_df.columns:
    if eda_df[col].dtype == object:
        eda_df[col] = pd.to_numeric(eda_df[col], errors="ignore")

numeric_columns = eda_df.select_dtypes(include="number").columns.tolist()
categorical_columns = [c for c in eda_df.columns if c not in numeric_columns]

print("Numeric columns:", numeric_columns)
print("Categorical/non-numeric columns:", categorical_columns)

In [ ]:
# Template-style numeric filtering, normalization, and grouping.
if numeric_columns:
    numeric_field_id = numeric_columns[0]
    threshold = eda_df[numeric_field_id].median()
    filtered_df = eda_df[eda_df[numeric_field_id] > threshold].copy()
    
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    std_value = filtered_df[numeric_field_id].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / std_value
    else:
        filtered_df[f"{numeric_field_id}_normalized"] = 0.0
    
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    group_field_id = categorical_columns[0] if categorical_columns else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean(numeric_only=True).reset_index()
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        grouped_df = None
        print("No categorical group field available for grouping.")
else:
    print("No numeric columns were found. Running metadata-oriented EDA instead.")
    
    # Count lengths of string representations to create a useful derived feature
    metadata_eda_df = eda_df.copy()
    if "value" in metadata_eda_df.columns:
        metadata_eda_df["value_length"] = metadata_eda_df["value"].astype(str).str.len()
        numeric_field_id = "value_length"
        threshold = metadata_eda_df[numeric_field_id].median()
        filtered_df = metadata_eda_df[metadata_eda_df[numeric_field_id] > threshold].copy()
        print(f"Filtered metadata rows with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field_id].std()
        if pd.notna(std_value) and std_value != 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / std_value
        else:
            filtered_df[f"{numeric_field_id}_normalized"] = 0.0
        
        print(f"Normalized {numeric_field_id} for filtered metadata rows:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        group_field_id = "attribute_@id" if "attribute_@id" in filtered_df.columns else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            grouped_df = None
    else:
        filtered_df = metadata_eda_df.copy()
        grouped_df = None
        numeric_field_id = None
        print("No 'value' column was available for fallback metadata EDA.")

In [ ]:
# Additional dataset-level exploration using metadata_json
dataset_level_summary = {
    "n_authors": len(metadata_json.get("author", []) or []),
    "n_citations": len(metadata_json.get("citation", []) or []),
    "n_keywords": len(metadata_json.get("keywords", []) or []),
    "n_data_use_cases": len(metadata_json.get("dataUseCases", []) or []),
    "n_data_limitations": len(metadata_json.get("dataLimitations", []) or []),
    "n_data_biases": len(metadata_json.get("dataBiases", []) or []),
    "n_distributions": len(metadata_json.get("distribution", []) or []),
    "n_record_sets": len(metadata_json.get("recordSet", []) or []),
    "n_visualizations": len(metadata_json.get("visualization", []) or []),
}

dataset_level_summary_df = pd.DataFrame(
    [{"metric_@id": k, "value": v} for k, v in dataset_level_summary.items()]
)
dataset_level_summary_df

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

This section adapts automatically based on whether tabular numeric fields are available. If not, it visualizes dataset-level metadata counts and derived metadata lengths.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=dataset_level_summary_df, x="metric_@id", y="value", palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("Dataset-level metadata counts")
plt.xlabel("Metric @id")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
if numeric_columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(eda_df[numeric_columns[0]].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of numeric field: {numeric_columns[0]}")
    plt.xlabel(numeric_columns[0])
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    if "value" in dataframes[selected_record_set_id].columns:
        temp_df = dataframes[selected_record_set_id].copy()
        temp_df["value_length"] = temp_df["value"].astype(str).str.len()
        plt.figure(figsize=(8, 5))
        sns.histplot(temp_df["value_length"], kde=True, bins=15)
        plt.title("Distribution of metadata value lengths")
        plt.xlabel("value_length")
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()
    else:
        print("No suitable column was available for a distribution plot.")

In [ ]:
if grouped_df is not None and not grouped_df.empty:
    x_col = grouped_df.columns[0]
    y_col = grouped_df.columns[1]
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=x_col, y=y_col, palette="magma")
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Grouped summary by {x_col}")
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.tight_layout()
    plt.show()
else:
    print("Grouped visualization skipped because no grouped DataFrame was available.")

## 6. Conclusion
This notebook demonstrated how to:

- Load a Croissant dataset from a schema URL with `mlcroissant`.
- Access metadata safely through `dataset.metadata` as a single object.
- Discover record sets and fields using their `@id` values.
- Extract record-set data into pandas DataFrames when available.
- Fall back to metadata-centric analysis when the dataset does not expose tabular record sets.
- Perform basic EDA, filtering, normalization, grouping, and visualization in a way that adapts to the dataset structure.

For this ETU TORUS dataset, the provided FAIR metadata suggests a strong focus on provenance, bibliographic description, file-level integrity information, and documentary traceability. If future versions of the Croissant resource expose one or more populated `recordSet` entries, the same notebook structure can be reused directly by selecting those record set `@id` values.